# Geometry-V1 Batch 2B
PREPARED_NOT_EXECUTED. User-run Colab only; no current push, Colab, Drive, or model-execution authorization.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
import hashlib,json,os,pathlib,shutil,subprocess,sys,zipfile
from google.colab import files,userdata
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'; BRANCH='Geometry-V1'
EXECUTION_EXACT='f1b89c00f19fa561235170aaeb342671a69c5906'; RUN_ID='geometry-v1-b2b-f1b89c00f19f-operational-01'
PROPOSED_PENDING_FINAL_USER_CONFIRMATION="/content/drive/MyDrive/CEG-WM/Geometry-V1/Batch2B"
DRIVE_ROOT=pathlib.Path(PROPOSED_PENDING_FINAL_USER_CONFIRMATION); repo=pathlib.Path('/content/geometry-v1-source'); run_dir=DRIVE_ROOT/RUN_ID
root_key=''; hf_token=''; runner_env=None; process=None; uploaded_paths=[]
input_dir=pathlib.Path('/content/geometry-v1-inputs'); work=pathlib.Path('/content/geometry-v1-receipt'); archive=pathlib.Path('/content')/(RUN_ID+'.zip'); sidecar=pathlib.Path('/content')/(RUN_ID+'.zip.sha256')
def verify_checkout():
 h=subprocess.run(['git','rev-parse','HEAD'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip(); c=subprocess.run(['git','status','--porcelain'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip()
 if h!=EXECUTION_EXACT or c: raise RuntimeError('checkout identity differs')
def exclusive_copy(source,target):
 with source.open('rb') as read, target.open('xb') as write:
  while chunk:=read.read(1048576): write.write(chunk)
def parse_child(rc,stdout):
 if len(stdout)>4096: raise RuntimeError('bounded stdout exceeded')
 lines=stdout.decode('utf-8','strict').splitlines(); success='CEGWM_GEOMETRY_V1_OPERATIONAL_PREFLIGHT '; failure='CEGWM_GEOMETRY_V1_OPERATIONAL_FAILURE '
 if len(lines)!=1 or not lines[0]: raise RuntimeError('exactly one receipt line required')
 if lines[0].startswith(success): prefix,status=success,'success'
 elif lines[0].startswith(failure): prefix,status=failure,'failure'
 else: raise RuntimeError('invalid receipt prefix')
 payload=json.loads(lines[0][len(prefix):])
 if not isinstance(payload,dict) or (rc==0)!=(status=='success'): raise RuntimeError('receipt/return-code mismatch')
 return status,payload,lines[0]
try:
 if repo.exists() or run_dir.exists() or input_dir.exists() or work.exists() or archive.exists() or sidecar.exists(): raise FileExistsError('create-only path exists')
 subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
 subprocess.run(['git','checkout','--detach',EXECUTION_EXACT],cwd=repo,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); verify_checkout()
 subprocess.run([sys.executable,'-m','pip','install',str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); verify_checkout()
 uploaded=files.upload()
 if len(uploaded) not in (1,2): raise ValueError('upload exactly one or two ordinary RGB images')
 input_dir.mkdir()
 for name,data in uploaded.items():
  path=input_dir/pathlib.Path(name).name
  with path.open('xb') as h: h.write(data)
  uploaded_paths.append(path)
 hf_token=userdata.get('HF_TOKEN'); root_key=userdata.get('CEG_WM_ROOT_KEY')
 if not hf_token or not root_key: raise RuntimeError('required secret unavailable')
 runner_env={n:v for n,v in os.environ.items() if all(x not in n.upper() for x in ('TOKEN','KEY','SECRET'))}; runner_env['HF_TOKEN']=hf_token; runner_env['CEG_WM_ROOT_KEY']=root_key
 command=[sys.executable,'-m','experiments.run_geometry_v1_qk_operational_preflight','--repo-root',str(repo),'--expected-exact',EXECUTION_EXACT,*map(str,uploaded_paths)]
 process=subprocess.Popen(command,cwd=repo,env=runner_env,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL); stdout,_=process.communicate(timeout=1800); status,payload,line=parse_child(process.returncode,stdout)
 work.mkdir(); status_name='success.json' if status=='success' else 'failure.json'
 for name,value in [('receipt.json',payload),(status_name,{'status':payload.get('status')})]:
  with (work/name).open('x',encoding='utf-8') as h: json.dump(value,h,sort_keys=True,separators=(',',':'))
 allowed=['receipt.json',status_name,'manifest.json','SHA256SUMS']
 with (work/'manifest.json').open('x',encoding='utf-8') as h: json.dump({'execution_exact':EXECUTION_EXACT,'run_id':RUN_ID,'allowed_filenames':allowed},h,sort_keys=True,separators=(',',':'))
 with (work/'SHA256SUMS').open('x',encoding='ascii') as h:
  for name in allowed[:-1]: h.write(hashlib.sha256((work/name).read_bytes()).hexdigest()+'  '+name+'\n')
 with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED) as z:
  for name in allowed: z.write(work/name,name)
 with archive.open('rb') as h: digest=hashlib.sha256(h.read()).hexdigest()
 with sidecar.open('xb') as h: h.write((digest+'  '+archive.name+'\n').encode('ascii'))
 DRIVE_ROOT.mkdir(parents=True,exist_ok=True); run_dir.mkdir(); exclusive_copy(archive,run_dir/archive.name); exclusive_copy(sidecar,run_dir/sidecar.name)
 print(line)
 if status=='failure': raise RuntimeError('child reported sanitized failure')
finally:
 root_key=''; hf_token=''
 if runner_env is not None:
  runner_env.pop('HF_TOKEN',None); runner_env.pop('CEG_WM_ROOT_KEY',None)
 if process is not None and process.poll() is None: process.kill(); process.wait()
 for path in uploaded_paths:
  if path.exists(): path.unlink()
 if input_dir.exists(): input_dir.rmdir()
 for path in (work,archive,sidecar):
  if path.is_dir(): shutil.rmtree(path)
  elif path.exists(): path.unlink()
